# Chapter 3.9: Sampling & Logits Processing Pipeline in vLLM

## Learning Objectives

By the end of this notebook, you will:

1. Understand how vLLM transforms raw model logits into sampled tokens
2. Know the full LogitsProcessor chain: temperature, top-k, top-p, penalties
3. Understand the Sampler class architecture and sampling strategies
4. Be able to configure SamplingParams for different use cases
5. Know how best-of-n sampling works for quality improvement
6. Understand how structured output hooks integrate with the sampling pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part3/chapter_3.9_sampling_pipeline.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part3/chapter_3.9_sampling_pipeline.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. The Sampling Pipeline Overview

After the model produces raw logits for the next token, vLLM applies a series of transformations before selecting the final token.

```
Model Output (logits)
    │
    ▼
┌──────────────────────┐
│  LogitsProcessor      │  ← Chain of transformations
│  ├─ Penalties          │    (repetition, frequency, presence)
│  ├─ Temperature        │    (scale logits)
│  ├─ Top-k              │    (keep top k tokens)
│  ├─ Top-p (nucleus)    │    (keep top cumulative p)
│  ├─ Min-p              │    (keep tokens above min_p * max_prob)
│  └─ Custom processors  │    (structured output, grammar, etc.)
├──────────────────────┤
│  Sampler              │  ← Select token(s)
│  ├─ Greedy            │    (argmax)
│  ├─ Random            │    (multinomial from distribution)
│  └─ Beam search       │    (maintain top beams)
└──────────────────────┘
    │
    ▼
Selected Token(s)
```

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Figure: Logits Processing Pipeline ──
# Raw logits -> Temperature -> Top-k mask -> Top-p mask -> Penalties -> Softmax -> Sample
BLUE = '#4A90D9'; GREEN = '#27AE60'; ORANGE = '#F39C12'; RED = '#E74C3C'; PURPLE = '#8E44AD'

fig, ax = plt.subplots(figsize=(18, 4.5))
ax.set_xlim(0, 18); ax.set_ylim(0, 4.5); ax.axis('off')
fig.patch.set_facecolor('white')

stages = [
    (1.2,  'Raw\nLogits',      '#95A5A6'),
    (3.5,  'Penalties\n(rep/freq/pres)', RED),
    (6.0,  'Temperature\nScaling',       ORANGE),
    (8.5,  'Top-k\nMask',               BLUE),
    (11.0, 'Top-p\n(Nucleus)',           BLUE),
    (13.5, 'Softmax',                    PURPLE),
    (16.0, 'Sample\nToken',              GREEN),
]

bw, bh = 2.0, 1.8
for x, label, color in stages:
    rect = mpatches.FancyBboxPatch((x - bw/2, 2.5 - bh/2), bw, bh,
        boxstyle="round,pad=0.12", facecolor=color, edgecolor='white',
        linewidth=2, alpha=0.9)
    ax.add_patch(rect)
    ax.text(x, 2.5, label, ha='center', va='center', fontsize=9,
            fontweight='bold', color='white')

# Arrows
for i in range(len(stages) - 1):
    sx = stages[i][0] + bw/2
    dx = stages[i+1][0] - bw/2
    ax.annotate('', xy=(dx, 2.5), xytext=(sx, 2.5),
                arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2.5))

# Annotations below
notes = [
    (1.2,  '[V]-dim\nvector'),
    (3.5,  'Reduce repeated\ntoken logits'),
    (6.0,  'logits / T\n(sharpen/flatten)'),
    (8.5,  'Keep top-k,\nrest = -inf'),
    (11.0, 'Keep smallest set\nwith cum_p >= p'),
    (13.5, 'exp(x) / sum\n-> probabilities'),
    (16.0, 'argmax (greedy)\nor multinomial'),
]
for x, text in notes:
    ax.text(x, 0.5, text, ha='center', va='center', fontsize=7,
            color='#555', style='italic')

ax.set_title('Logits Processing Pipeline: From Raw Model Output to Sampled Token',
             fontsize=14, fontweight='bold', pad=8)
plt.tight_layout(); plt.show()

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Optional, Dict, Callable
from dataclasses import dataclass, field
import time
import json

print("Imports ready.")

## 2. SamplingParams — The Configuration Object

In [ ]:
# Source: vllm/sampling_params.py (simplified)

@dataclass
class SamplingParams:
    """Parameters for the sampling process.
    
    This is the main user-facing configuration object.
    Every request carries a SamplingParams instance.
    """
    # Sampling strategy
    temperature: float = 1.0        # 0 = greedy, >0 = random
    top_k: int = -1                 # -1 = disabled
    top_p: float = 1.0              # 1.0 = disabled (nucleus sampling)
    min_p: float = 0.0              # 0.0 = disabled
    
    # Penalties (reduce repetition)
    repetition_penalty: float = 1.0  # 1.0 = disabled
    frequency_penalty: float = 0.0   # 0.0 = disabled
    presence_penalty: float = 0.0    # 0.0 = disabled
    
    # Generation control
    max_tokens: int = 16             # Max tokens to generate
    stop: List[str] = field(default_factory=list)  # Stop strings
    seed: Optional[int] = None       # For reproducibility
    
    # Best-of-n
    n: int = 1                       # Number of outputs to return
    best_of: Optional[int] = None    # Generate best_of, return top n
    
    # Logprobs
    logprobs: Optional[int] = None   # Return top-k logprobs
    prompt_logprobs: Optional[int] = None
    
    def __post_init__(self):
        """Validate parameters."""
        if self.temperature < 0:
            raise ValueError("temperature must be >= 0")
        if self.top_p < 0 or self.top_p > 1:
            raise ValueError("top_p must be in [0, 1]")
        if self.best_of is not None and self.best_of < self.n:
            raise ValueError("best_of must be >= n")

# Example configurations
greedy = SamplingParams(temperature=0)
creative = SamplingParams(temperature=0.9, top_p=0.95)
strict = SamplingParams(temperature=0.3, top_k=10)
diverse = SamplingParams(temperature=1.0, top_p=0.9, repetition_penalty=1.2)

print("Greedy:  ", greedy)
print("Creative:", creative)
print("Strict:  ", strict)
print("Diverse: ", diverse)

## 3. Temperature Scaling

In [ ]:
def apply_temperature(logits: torch.Tensor, temperature: float) -> torch.Tensor:
    """Scale logits by temperature.
    
    temperature < 1: sharper distribution (more deterministic)
    temperature = 1: unchanged
    temperature > 1: flatter distribution (more random)
    temperature = 0: greedy (argmax)
    """
    if temperature == 0:
        return logits  # Handled by greedy sampling
    return logits / temperature

# Demo: effect of temperature on probability distribution
vocab_size = 10
raw_logits = torch.tensor([5.0, 3.0, 2.0, 1.5, 1.0, 0.5, 0.2, 0.1, -0.5, -1.0])

temperatures = [0.1, 0.5, 1.0, 1.5, 2.0]
fig, axes = plt.subplots(1, len(temperatures), figsize=(18, 3.5), sharey=True)

for ax, temp in zip(axes, temperatures):
    scaled = apply_temperature(raw_logits, temp)
    probs = F.softmax(scaled, dim=-1).numpy()
    ax.bar(range(vocab_size), probs, color='steelblue')
    ax.set_title(f'T = {temp}')
    ax.set_xlabel('Token ID')
    ax.set_ylim(0, 1.05)
    if ax == axes[0]:
        ax.set_ylabel('Probability')

plt.suptitle('Effect of Temperature on Token Probabilities', fontweight='bold')
plt.tight_layout()
plt.show()

print("Low temperature (0.1): nearly greedy, picks the most likely token")
print("High temperature (2.0): almost uniform, very diverse/random output")

## 4. Top-k Filtering

In [ ]:
def apply_top_k(logits: torch.Tensor, top_k: int) -> torch.Tensor:
    """Keep only the top-k highest logits, set rest to -inf.
    
    This restricts sampling to the top_k most likely tokens.
    Prevents sampling from the long tail of unlikely tokens.
    """
    if top_k <= 0 or top_k >= logits.size(-1):
        return logits
    
    # Find the k-th largest value
    top_k_values, _ = torch.topk(logits, top_k, dim=-1)
    min_top_k = top_k_values[:, -1] if logits.dim() > 1 else top_k_values[-1]
    
    # Mask everything below the k-th largest
    if logits.dim() > 1:
        mask = logits < min_top_k.unsqueeze(-1)
    else:
        mask = logits < min_top_k
    return logits.masked_fill(mask, float('-inf'))

# Demo
raw = torch.tensor([5.0, 3.0, 2.0, 1.5, 1.0, 0.5, 0.2, 0.1, -0.5, -1.0])

for k in [3, 5, 10]:
    filtered = apply_top_k(raw.clone(), k)
    probs = F.softmax(filtered, dim=-1)
    active = (probs > 0).sum().item()
    print(f"top_k={k:>2}: active tokens = {active}, probs = {probs.numpy().round(4)}")

## 5. Top-p (Nucleus) Sampling

In [ ]:
def apply_top_p(logits: torch.Tensor, top_p: float) -> torch.Tensor:
    """Nucleus sampling: keep smallest set of tokens whose cumulative probability >= top_p.
    
    Unlike top-k which always keeps exactly k tokens, top-p adapts:
    - When model is confident: keeps fewer tokens
    - When model is uncertain: keeps more tokens
    """
    if top_p >= 1.0:
        return logits
    
    # Sort in descending order
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    sorted_probs = F.softmax(sorted_logits, dim=-1)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
    
    # Remove tokens with cumulative probability above threshold
    # Shift right so the first token that exceeds threshold is kept
    sorted_mask = cumulative_probs - sorted_probs > top_p
    sorted_logits[sorted_mask] = float('-inf')
    
    # Scatter back to original order
    result = torch.zeros_like(logits)
    result.scatter_(0, sorted_indices, sorted_logits)
    return result

# Demo: top-p adapts to confidence level
confident_logits = torch.tensor([10.0, 1.0, 0.5, 0.1, -1.0, -2.0, -3.0, -4.0, -5.0, -6.0])
uncertain_logits = torch.tensor([2.0, 1.9, 1.8, 1.7, 1.6, 1.5, 1.4, 1.3, 1.2, 1.1])

for name, logits in [("Confident", confident_logits), ("Uncertain", uncertain_logits)]:
    filtered = apply_top_p(logits.clone(), top_p=0.9)
    probs = F.softmax(filtered, dim=-1)
    active = (probs > 0).sum().item()
    print(f"{name:>10} model, top_p=0.9: {active} tokens kept")
    print(f"           probs: {probs.numpy().round(4)}")

## 6. Min-p Filtering

In [ ]:
def apply_min_p(logits: torch.Tensor, min_p: float) -> torch.Tensor:
    """Min-p filtering: keep tokens whose probability >= min_p * max_probability.
    
    A simpler alternative to top-p that also adapts to confidence.
    More intuitive: "keep tokens that are at least min_p fraction of the best token."
    """
    if min_p <= 0.0:
        return logits
    
    probs = F.softmax(logits, dim=-1)
    max_prob = probs.max(dim=-1, keepdim=True).values
    threshold = min_p * max_prob
    
    mask = probs < threshold
    return logits.masked_fill(mask, float('-inf'))

# Demo
raw = torch.tensor([5.0, 3.0, 2.0, 1.5, 1.0, 0.5, 0.2, 0.1, -0.5, -1.0])
probs_orig = F.softmax(raw, dim=-1)
print(f"Original probs: {probs_orig.numpy().round(4)}")
print(f"Max prob:        {probs_orig.max().item():.4f}\n")

for min_p_val in [0.05, 0.1, 0.2, 0.5]:
    filtered = apply_min_p(raw.clone(), min_p_val)
    probs_f = F.softmax(filtered, dim=-1)
    active = (probs_f > 0).sum().item()
    threshold = min_p_val * probs_orig.max().item()
    print(f"min_p={min_p_val:.2f} (threshold={threshold:.4f}): {active} tokens kept")

## 7. Repetition Penalties

In [ ]:
def apply_repetition_penalty(
    logits: torch.Tensor,
    output_token_ids: List[int],
    repetition_penalty: float = 1.0,
    frequency_penalty: float = 0.0,
    presence_penalty: float = 0.0,
) -> torch.Tensor:
    """Apply various repetition penalties to logits.
    
    Three types:
    1. repetition_penalty: Multiplicative penalty on previously seen tokens.
       logit = logit / penalty (if logit > 0) or logit * penalty (if logit < 0)
    
    2. frequency_penalty: Additive penalty proportional to token count.
       logit -= frequency_penalty * count(token)
    
    3. presence_penalty: Additive penalty for any previously seen token.
       logit -= presence_penalty * (1 if token seen else 0)
    """
    result = logits.clone()
    
    if not output_token_ids:
        return result
    
    # Count token frequencies
    token_counts: Dict[int, int] = {}
    for tok_id in output_token_ids:
        token_counts[tok_id] = token_counts.get(tok_id, 0) + 1
    
    for tok_id, count in token_counts.items():
        if tok_id >= result.size(-1):
            continue
        
        # Repetition penalty (multiplicative)
        if repetition_penalty != 1.0:
            if result[tok_id] > 0:
                result[tok_id] /= repetition_penalty
            else:
                result[tok_id] *= repetition_penalty
        
        # Frequency penalty (additive, proportional to count)
        result[tok_id] -= frequency_penalty * count
        
        # Presence penalty (additive, binary)
        result[tok_id] -= presence_penalty
    
    return result

# Demo: effect of penalties
vocab_size = 8
logits = torch.tensor([4.0, 3.0, 2.5, 2.0, 1.5, 1.0, 0.5, 0.0])
previously_generated = [0, 0, 0, 1, 1, 2]  # Token 0 appeared 3 times, token 1 twice, etc.

print("Original probs:")
print(f"  {F.softmax(logits, dim=-1).numpy().round(4)}\n")

configs = [
    {"repetition_penalty": 1.5, "frequency_penalty": 0.0, "presence_penalty": 0.0},
    {"repetition_penalty": 1.0, "frequency_penalty": 0.5, "presence_penalty": 0.0},
    {"repetition_penalty": 1.0, "frequency_penalty": 0.0, "presence_penalty": 1.0},
    {"repetition_penalty": 1.3, "frequency_penalty": 0.3, "presence_penalty": 0.5},
]

for config in configs:
    penalized = apply_repetition_penalty(logits, previously_generated, **config)
    probs = F.softmax(penalized, dim=-1).numpy().round(4)
    label = ", ".join(f"{k}={v}" for k, v in config.items())
    print(f"{label}")
    print(f"  probs: {probs}")

## 8. The LogitsProcessor Chain

In [ ]:
# Source: vllm/model_executor/layers/sampler.py (simplified)

class LogitsProcessor:
    """Base class for logits processors."""
    def __call__(self, logits: torch.Tensor, **kwargs) -> torch.Tensor:
        raise NotImplementedError

class TemperatureProcessor(LogitsProcessor):
    def __init__(self, temperature: float):
        self.temperature = temperature
    
    def __call__(self, logits, **kwargs):
        if self.temperature == 0:
            return logits
        return logits / self.temperature

class TopKProcessor(LogitsProcessor):
    def __init__(self, top_k: int):
        self.top_k = top_k
    
    def __call__(self, logits, **kwargs):
        return apply_top_k(logits, self.top_k)

class TopPProcessor(LogitsProcessor):
    def __init__(self, top_p: float):
        self.top_p = top_p
    
    def __call__(self, logits, **kwargs):
        return apply_top_p(logits, self.top_p)

class MinPProcessor(LogitsProcessor):
    def __init__(self, min_p: float):
        self.min_p = min_p
    
    def __call__(self, logits, **kwargs):
        return apply_min_p(logits, self.min_p)

class PenaltyProcessor(LogitsProcessor):
    def __init__(self, rep_pen, freq_pen, pres_pen):
        self.rep_pen = rep_pen
        self.freq_pen = freq_pen
        self.pres_pen = pres_pen
    
    def __call__(self, logits, output_token_ids=None, **kwargs):
        if output_token_ids is None:
            output_token_ids = []
        return apply_repetition_penalty(
            logits, output_token_ids,
            self.rep_pen, self.freq_pen, self.pres_pen
        )


def build_logits_processors(params: SamplingParams) -> List[LogitsProcessor]:
    """Build the chain of logits processors from SamplingParams.
    
    Order matters! vLLM applies them in this order:
    1. Penalties (on raw logits)
    2. Temperature (scale)
    3. Top-k (hard cutoff)
    4. Top-p / Min-p (adaptive cutoff)
    """
    processors = []
    
    # 1. Penalties first (operate on raw logits)
    if (params.repetition_penalty != 1.0 or 
        params.frequency_penalty != 0.0 or 
        params.presence_penalty != 0.0):
        processors.append(PenaltyProcessor(
            params.repetition_penalty,
            params.frequency_penalty,
            params.presence_penalty,
        ))
    
    # 2. Temperature
    if params.temperature != 1.0 and params.temperature != 0:
        processors.append(TemperatureProcessor(params.temperature))
    
    # 3. Top-k
    if params.top_k > 0:
        processors.append(TopKProcessor(params.top_k))
    
    # 4. Top-p
    if params.top_p < 1.0:
        processors.append(TopPProcessor(params.top_p))
    
    # 5. Min-p
    if params.min_p > 0.0:
        processors.append(MinPProcessor(params.min_p))
    
    return processors


# Demo
params = SamplingParams(temperature=0.7, top_k=5, top_p=0.9, repetition_penalty=1.2)
chain = build_logits_processors(params)
print(f"Processor chain for {params}:")
for i, p in enumerate(chain):
    print(f"  {i+1}. {p.__class__.__name__}")

In [ ]:
# Run the full chain on example logits

def run_logits_pipeline(logits: torch.Tensor, processors: List[LogitsProcessor], 
                        output_token_ids: List[int] = None):
    """Run the full logits processing pipeline, printing intermediate results."""
    current = logits.clone()
    print(f"{'Stage':<25} {'Active Tokens':>14} {'Top-3 Probs':>30}")
    print("=" * 70)
    
    probs = F.softmax(current, dim=-1)
    top3 = torch.topk(probs, 3)
    active = (probs > 0).sum().item()
    top3_str = ", ".join(f"{v:.4f}" for v in top3.values)
    print(f"{'Raw logits':<25} {active:>14} {top3_str:>30}")
    
    for proc in processors:
        current = proc(current, output_token_ids=output_token_ids or [])
        probs = F.softmax(current, dim=-1)
        active = (probs > 1e-8).sum().item()
        top3 = torch.topk(probs, min(3, active))
        top3_str = ", ".join(f"{v:.4f}" for v in top3.values)
        print(f"{proc.__class__.__name__:<25} {active:>14} {top3_str:>30}")
    
    return current

logits = torch.randn(50)  # 50-token vocabulary
logits[0] = 5.0   # Make a few tokens dominant
logits[1] = 3.5
logits[2] = 3.0

params = SamplingParams(temperature=0.8, top_k=10, top_p=0.9, repetition_penalty=1.3)
processors = build_logits_processors(params)
prev_tokens = [0, 0, 1, 3, 5]

print("Pipeline with repetition penalty + temperature + top-k + top-p:\n")
final_logits = run_logits_pipeline(logits, processors, prev_tokens)

## 9. The Sampler Class

In [ ]:
# Source: vllm/model_executor/layers/sampler.py (simplified)

@dataclass
class SamplerOutput:
    """Output from the sampler."""
    token_ids: List[int]        # Selected token IDs
    logprobs: Optional[List[Dict[int, float]]] = None  # Log-probs for top tokens


class Sampler:
    """The main sampler that selects tokens from processed logits.
    
    In vLLM, this handles batched sampling across multiple sequences
    with potentially different SamplingParams.
    """
    
    def sample(self, logits: torch.Tensor, params: SamplingParams,
               generators: Optional[Dict[int, torch.Generator]] = None) -> SamplerOutput:
        """Sample tokens from processed logits."""
        
        if params.temperature == 0:
            # Greedy: argmax
            token_ids = self._greedy_sample(logits)
        else:
            # Random: multinomial sampling
            token_ids = self._random_sample(logits, params, generators)
        
        # Compute logprobs if requested
        logprobs = None
        if params.logprobs is not None:
            logprobs = self._compute_logprobs(logits, token_ids, params.logprobs)
        
        return SamplerOutput(token_ids=token_ids, logprobs=logprobs)
    
    def _greedy_sample(self, logits: torch.Tensor) -> List[int]:
        """Greedy sampling: pick the highest probability token."""
        return logits.argmax(dim=-1).tolist()
    
    def _random_sample(self, logits: torch.Tensor, params: SamplingParams,
                       generators=None) -> List[int]:
        """Random sampling: sample from the probability distribution."""
        probs = F.softmax(logits, dim=-1)
        
        if logits.dim() == 1:
            logits = logits.unsqueeze(0)
            probs = probs.unsqueeze(0)
        
        # Use seed-based generator for reproducibility
        if params.seed is not None and generators is None:
            g = torch.Generator()
            g.manual_seed(params.seed)
            token_ids = torch.multinomial(probs, num_samples=1, generator=g)
        else:
            token_ids = torch.multinomial(probs, num_samples=1)
        
        return token_ids.squeeze(-1).tolist()
    
    def _compute_logprobs(self, logits, token_ids, num_logprobs):
        """Compute log-probabilities for the top tokens."""
        log_probs = F.log_softmax(logits, dim=-1)
        if logits.dim() == 1:
            log_probs = log_probs.unsqueeze(0)
        
        results = []
        for i, tok_id in enumerate(token_ids):
            top_vals, top_ids = torch.topk(log_probs[i], num_logprobs)
            entry = {tid.item(): val.item() for tid, val in zip(top_ids, top_vals)}
            entry[tok_id] = log_probs[i, tok_id].item()
            results.append(entry)
        return results

# Demo
sampler = Sampler()

logits = torch.tensor([5.0, 3.0, 2.0, 1.0, 0.5, 0.1, -1.0, -2.0])

# Greedy
result = sampler.sample(logits, SamplingParams(temperature=0))
print(f"Greedy: token_id = {result.token_ids}")

# Random with seed for reproducibility
result = sampler.sample(logits, SamplingParams(temperature=0.8, seed=42, logprobs=3))
print(f"Random (seed=42): token_id = {result.token_ids}")
print(f"  logprobs: {result.logprobs}")

## 10. Best-of-N Sampling

In [ ]:
def best_of_n_sample(
    generate_fn: Callable,
    prompt: str,
    n: int,
    best_of: int,
    scoring_fn: Callable = None,
) -> List[str]:
    """Generate `best_of` completions and return the top `n`.
    
    In vLLM:
    - Generates `best_of` independent sequences in parallel
    - Scores each by cumulative log-probability
    - Returns the top `n` sequences
    
    This trades compute for quality: more candidates = better best output.
    """
    if scoring_fn is None:
        # Default: score by average log-probability (perplexity)
        scoring_fn = lambda text, logprob: logprob / max(len(text.split()), 1)
    
    # Generate best_of candidates
    candidates = []
    for i in range(best_of):
        text, logprob = generate_fn(prompt, seed=i)
        score = scoring_fn(text, logprob)
        candidates.append((text, score, logprob))
    
    # Sort by score (higher is better)
    candidates.sort(key=lambda x: x[1], reverse=True)
    
    return candidates[:n]


# Simulate best-of-n
def fake_generate(prompt, seed=0):
    np.random.seed(seed)
    words = ["the", "cat", "sat", "on", "a", "mat", "dog", "ran", "quickly", "slowly"]
    length = np.random.randint(3, 8)
    text = " ".join(np.random.choice(words, length))
    logprob = np.random.uniform(-5, -1)
    return text, logprob

print("Best-of-5, return top 2:\n")
results = best_of_n_sample(fake_generate, "Once upon", n=2, best_of=5)
for i, (text, score, logprob) in enumerate(results):
    print(f"  #{i+1}: score={score:.4f}, logprob={logprob:.4f}, text=\"{text}\"")

print("\nBest-of-10, return top 1:\n")
results = best_of_n_sample(fake_generate, "Once upon", n=1, best_of=10)
for i, (text, score, logprob) in enumerate(results):
    print(f"  #{i+1}: score={score:.4f}, logprob={logprob:.4f}, text=\"{text}\"")

## 11. Custom LogitsProcessor

In [ ]:
# Demo 1: Build and test a custom logits processor

class BannedTokensProcessor(LogitsProcessor):
    """Ban specific tokens from being generated."""
    def __init__(self, banned_token_ids: List[int]):
        self.banned_token_ids = banned_token_ids
    
    def __call__(self, logits, **kwargs):
        result = logits.clone()
        for tok_id in self.banned_token_ids:
            if tok_id < result.size(-1):
                result[tok_id] = float('-inf')
        return result


class BiasTokensProcessor(LogitsProcessor):
    """Add bias to specific tokens (logit_bias in OpenAI API)."""
    def __init__(self, biases: Dict[int, float]):
        self.biases = biases
    
    def __call__(self, logits, **kwargs):
        result = logits.clone()
        for tok_id, bias in self.biases.items():
            if tok_id < result.size(-1):
                result[tok_id] += bias
        return result


class MaxNewTokensProcessor(LogitsProcessor):
    """Force EOS token after max tokens are generated."""
    def __init__(self, max_tokens: int, eos_token_id: int):
        self.max_tokens = max_tokens
        self.eos_token_id = eos_token_id
    
    def __call__(self, logits, output_token_ids=None, **kwargs):
        if output_token_ids and len(output_token_ids) >= self.max_tokens:
            result = torch.full_like(logits, float('-inf'))
            result[self.eos_token_id] = 0.0
            return result
        return logits


# Test custom processors
logits = torch.tensor([5.0, 4.0, 3.0, 2.0, 1.0, 0.5, 0.1, -1.0])

# Ban tokens 0 and 1
ban = BannedTokensProcessor([0, 1])
result = ban(logits)
print("After banning tokens 0,1:")
print(f"  probs: {F.softmax(result, dim=-1).numpy().round(4)}\n")

# Bias token 5 upward
bias = BiasTokensProcessor({5: 10.0})
result = bias(logits)
print("After biasing token 5 by +10:")
print(f"  probs: {F.softmax(result, dim=-1).numpy().round(4)}\n")

# Force EOS after 3 tokens
eos = MaxNewTokensProcessor(max_tokens=3, eos_token_id=7)
result = eos(logits, output_token_ids=[2, 3, 4])  # 3 tokens generated
print("After 3 tokens generated (force EOS=7):")
print(f"  probs: {F.softmax(result, dim=-1).numpy().round(4)}")

## 12. Structured Output Hooks

In [ ]:
# Structured output: constrain generation to valid JSON, regex, grammar

class JSONModeProcessor(LogitsProcessor):
    """Simplified JSON mode: ensure output is valid JSON.
    
    In real vLLM, this uses outlines/lm-format-enforcer for grammar-guided generation.
    Here we show a simplified version that tracks JSON state.
    """
    
    # Simplified token mapping for demonstration
    TOKEN_MAP = {
        0: '{', 1: '}', 2: '[', 3: ']', 4: '"', 5: ':', 6: ',',
        7: 'true', 8: 'false', 9: 'null',
        10: '0', 11: '1', 12: '2', 13: 'a', 14: 'b', 15: 'c',
    }
    
    def __init__(self):
        self.state = 'START'  # Track JSON parsing state
        self.depth = 0
    
    def get_valid_tokens(self) -> List[int]:
        """Return token IDs valid at current position."""
        if self.state == 'START':
            return [0, 2]  # Must start with { or [
        elif self.state == 'OBJECT_KEY':
            return [4]  # String key
        elif self.state == 'OBJECT_COLON':
            return [5]  # :
        elif self.state == 'VALUE':
            return [0, 2, 4, 7, 8, 9, 10, 11, 12]  # Any value
        else:
            return list(range(16))  # All tokens
    
    def __call__(self, logits, **kwargs):
        valid = self.get_valid_tokens()
        mask = torch.full_like(logits, float('-inf'))
        for tok_id in valid:
            if tok_id < logits.size(-1):
                mask[tok_id] = logits[tok_id]
        return mask


# Demo
processor = JSONModeProcessor()
logits = torch.randn(16)

constrained = processor(logits)
probs = F.softmax(constrained, dim=-1)
print("JSON Mode at START state:")
print(f"  Valid tokens: { {i: processor.TOKEN_MAP[i] for i in processor.get_valid_tokens()} }")
print(f"  Probabilities: {probs.numpy().round(4)}")
print(f"  Only tokens 0 ('{{\'') and 2 ('[') have non-zero probability")

print("\n--- In real vLLM ---")
print("Structured output is handled by:")
print("  1. outlines: regex/JSON schema constrained generation")
print("  2. lm-format-enforcer: grammar-guided generation")
print("  3. xgrammar: context-free grammar engine")
print("These hook into the logits processing chain as custom processors.")

## 13. Demo: Sampling Strategy Comparison

In [ ]:
# Demo 2: Compare different sampling strategies on the same logits

def simulate_generation(logits: torch.Tensor, params: SamplingParams, 
                         num_steps: int = 20, vocab_size: int = 50) -> List[int]:
    """Simulate autoregressive generation with given sampling params."""
    sampler = Sampler()
    processors = build_logits_processors(params)
    
    generated = []
    for step in range(num_steps):
        # Simulate model output (base logits + small noise per step)
        step_logits = logits.clone() + torch.randn(vocab_size) * 0.5
        
        # Apply processors
        current = step_logits
        for proc in processors:
            current = proc(current, output_token_ids=generated)
        
        # Sample
        result = sampler.sample(current, params)
        generated.extend(result.token_ids)
    
    return generated


# Create base logits with a few dominant tokens
vocab_size = 50
base_logits = torch.randn(vocab_size) * 0.5
base_logits[0] = 5.0  # Very likely
base_logits[1] = 3.5
base_logits[2] = 3.0
base_logits[3] = 2.5
base_logits[4] = 2.0

strategies = {
    "Greedy (T=0)": SamplingParams(temperature=0),
    "Low-T (T=0.3)": SamplingParams(temperature=0.3),
    "Medium-T (T=0.7, top_p=0.9)": SamplingParams(temperature=0.7, top_p=0.9),
    "High-T (T=1.0, top_k=10)": SamplingParams(temperature=1.0, top_k=10),
    "Creative (T=1.2, top_p=0.95)": SamplingParams(temperature=1.2, top_p=0.95),
}

fig, axes = plt.subplots(1, len(strategies), figsize=(20, 4), sharey=True)

for ax, (name, params) in zip(axes, strategies.items()):
    tokens = simulate_generation(base_logits, params, num_steps=100, vocab_size=vocab_size)
    unique = len(set(tokens))
    
    # Plot token frequency histogram
    ax.hist(tokens, bins=range(vocab_size+1), color='steelblue', alpha=0.7, edgecolor='white')
    ax.set_title(f"{name}\n({unique} unique tokens)", fontsize=9)
    ax.set_xlabel('Token ID')
    if ax == axes[0]:
        ax.set_ylabel('Frequency')

plt.suptitle('Token Distribution by Sampling Strategy (100 steps)', fontweight='bold')
plt.tight_layout()
plt.show()

## 14. Demo: Repetition Penalty Effect

In [ ]:
# Demo 3: Visualize how repetition penalty affects diversity over time

def measure_diversity(params: SamplingParams, steps: int = 50, 
                       vocab_size: int = 30) -> List[float]:
    """Track unique token ratio over generation steps."""
    sampler = Sampler()
    processors = build_logits_processors(params)
    
    base_logits = torch.randn(vocab_size) * 0.3
    base_logits[0] = 4.0
    base_logits[1] = 3.0
    base_logits[2] = 2.5
    
    generated = []
    diversity = []
    
    for step in range(steps):
        step_logits = base_logits.clone() + torch.randn(vocab_size) * 0.2
        current = step_logits
        for proc in processors:
            current = proc(current, output_token_ids=generated)
        
        result = sampler.sample(current, params)
        generated.extend(result.token_ids)
        diversity.append(len(set(generated)) / len(generated))
    
    return diversity


penalty_configs = {
    "No penalty": SamplingParams(temperature=0.7),
    "rep_pen=1.2": SamplingParams(temperature=0.7, repetition_penalty=1.2),
    "rep_pen=1.5": SamplingParams(temperature=0.7, repetition_penalty=1.5),
    "freq_pen=0.5": SamplingParams(temperature=0.7, frequency_penalty=0.5),
    "pres_pen=1.0": SamplingParams(temperature=0.7, presence_penalty=1.0),
}

plt.figure(figsize=(10, 5))
for name, params in penalty_configs.items():
    diversity = measure_diversity(params, steps=50)
    plt.plot(diversity, label=name, linewidth=2)

plt.xlabel('Generation Step')
plt.ylabel('Unique Token Ratio')
plt.title('Token Diversity Over Generation Steps', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()

print("Higher unique ratio = more diverse generation")
print("Frequency penalty grows with count, so it prevents repetition more aggressively")
print("Presence penalty is binary: just discourages any re-use")

## 15. Batched Sampling in vLLM

In [ ]:
# In vLLM, the sampler processes an entire batch at once.
# Each sequence in the batch can have DIFFERENT SamplingParams.

batched_sampling = '''
class BatchedSampler:
    """How vLLM actually samples: batched across sequences.
    
    Key challenge: each sequence may have different:
    - temperature
    - top_k / top_p
    - penalties
    - sampling strategy (greedy vs random)
    """
    
    def forward(self, logits: torch.Tensor, 
                sampling_metadata: SamplingMetadata):
        # logits: [num_tokens, vocab_size]
        # Each token belongs to a sequence with its own SamplingParams
        
        # Step 1: Apply penalties (per-sequence, based on token history)
        logits = self._apply_penalties(
            logits, 
            sampling_metadata.output_token_ids,  # per-sequence history
            sampling_metadata.seq_groups,
        )
        
        # Step 2: Apply temperature (vectorized across batch)
        # Build a temperature tensor: [num_tokens, 1]
        temperatures = torch.tensor([
            sg.sampling_params.temperature 
            for sg in sampling_metadata.seq_groups
            for _ in sg.seq_ids  # expand for multi-sequence groups
        ]).unsqueeze(-1)
        logits = logits / temperatures.clamp(min=1e-8)
        
        # Step 3: Apply top-k and top-p (also batched)
        logits = self._apply_top_k_top_p(
            logits, sampling_metadata
        )
        
        # Step 4: Sample
        # Split into greedy vs random groups
        greedy_mask = temperatures.squeeze() == 0
        
        sampled = torch.empty(logits.size(0), dtype=torch.long)
        if greedy_mask.any():
            sampled[greedy_mask] = logits[greedy_mask].argmax(dim=-1)
        if (~greedy_mask).any():
            probs = F.softmax(logits[~greedy_mask], dim=-1)
            sampled[~greedy_mask] = torch.multinomial(probs, 1).squeeze(-1)
        
        return sampled
'''
print(batched_sampling)

In [ ]:
# Demonstrate batched sampling with different params per sequence

batch_size = 4
vocab_size = 20

# Simulated batch logits
batch_logits = torch.randn(batch_size, vocab_size)
# Make token 0 dominant for all
batch_logits[:, 0] = 5.0
batch_logits[:, 1] = 3.0

# Each sequence has different params
batch_params = [
    SamplingParams(temperature=0),        # Greedy
    SamplingParams(temperature=0.5),       # Conservative
    SamplingParams(temperature=1.0),       # Balanced
    SamplingParams(temperature=1.5),       # Creative
]

sampler = Sampler()
print(f"{'Sequence':>10} {'Strategy':>15} {'Sampled Token':>15} {'Top-3 Probs':>35}")
print("=" * 80)

for i in range(batch_size):
    seq_logits = batch_logits[i]
    params = batch_params[i]
    
    # Apply temperature
    if params.temperature > 0:
        processed = seq_logits / params.temperature
    else:
        processed = seq_logits
    
    result = sampler.sample(processed, params)
    probs = F.softmax(processed, dim=-1)
    top3 = torch.topk(probs, 3)
    top3_str = ", ".join(f"[{idx}]={v:.3f}" for idx, v in zip(top3.indices, top3.values))
    
    strategy = "greedy" if params.temperature == 0 else f"T={params.temperature}"
    print(f"{i:>10} {strategy:>15} {result.token_ids[0]:>15} {top3_str:>35}")

## 16. vLLM Sampler Source Architecture

In [ ]:
# Source architecture of the vLLM sampler

architecture = """
vLLM Sampling Architecture
==========================

File: vllm/model_executor/layers/sampler.py

class Sampler(nn.Module):
    ├── forward(logits, sampling_metadata)
    │   ├── _apply_logits_processors()       # Custom/structured output
    │   ├── _apply_penalties()                # rep/freq/pres penalties
    │   ├── _apply_top_k_top_p()             # Filtering
    │   ├── _apply_min_p()                    # Min-p filtering
    │   ├── _sample()                         # Greedy or multinomial
    │   ├── _get_logprobs()                   # Compute log-probs
    │   └── _build_sampler_output()           # Package results
    │
    └── Uses:
        ├── SamplingMetadata                  # Per-sequence metadata
        │   ├── seq_groups                    # Groups of sequences
        │   ├── selected_token_indices        # Which logits to sample from
        │   ├── categorized_sample_indices    # Greedy vs random split
        │   └── output_token_ids              # Previous tokens (for penalties)
        │
        ├── SamplingParams                    # User configuration
        │   ├── temperature, top_k, top_p
        │   ├── penalties (repetition, frequency, presence)
        │   ├── logprobs, seed, n, best_of
        │   └── logits_processors             # Custom hooks!
        │
        └── SamplerOutput                     # Results
            ├── outputs: List[CompletionOutput]
            └── each with: token_id, logprobs, text

Key Design Decisions:
  1. Penalties applied BEFORE temperature (on raw logits)
  2. Top-k applied BEFORE top-p (hard cutoff first)
  3. Greedy and random sequences batched separately
  4. Logprobs computed on final (post-processor) distribution
  5. Custom processors inserted via SamplingParams.logits_processors
"""
print(architecture)

## 17. Seed-Based Reproducibility

In [ ]:
# vLLM supports per-request seeds for reproducibility

def demonstrate_seeded_sampling():
    """Show that same seed produces same output."""
    logits = torch.randn(100)  # 100-token vocab
    logits[0] = 3.0
    logits[5] = 2.5
    logits[10] = 2.0
    
    sampler = Sampler()
    params_seeded = SamplingParams(temperature=0.8, seed=42)
    params_unseeded = SamplingParams(temperature=0.8)
    
    # Seeded: same result every time
    print("Seeded (seed=42) - 5 runs:")
    for i in range(5):
        result = sampler.sample(logits, params_seeded)
        print(f"  Run {i+1}: token = {result.token_ids[0]}")
    
    print("\nUnseeded - 5 runs:")
    for i in range(5):
        result = sampler.sample(logits, params_unseeded)
        print(f"  Run {i+1}: token = {result.token_ids[0]}")

demonstrate_seeded_sampling()

## 18. Sampling Performance Considerations

In [ ]:
# Benchmark: sampling overhead relative to model forward pass

def benchmark_sampling(vocab_size: int, batch_size: int, num_runs: int = 1000):
    """Measure sampling latency for different configurations."""
    logits = torch.randn(batch_size, vocab_size)
    
    configs = {
        "Greedy": SamplingParams(temperature=0),
        "Random (T=0.7)": SamplingParams(temperature=0.7),
        "Top-k=50": SamplingParams(temperature=0.7, top_k=50),
        "Top-p=0.9": SamplingParams(temperature=0.7, top_p=0.9),
        "Full pipeline": SamplingParams(
            temperature=0.7, top_k=50, top_p=0.9,
            repetition_penalty=1.2, frequency_penalty=0.3
        ),
    }
    
    results = {}
    for name, params in configs.items():
        processors = build_logits_processors(params)
        sampler = Sampler()
        prev_tokens = list(range(min(20, vocab_size)))
        
        start = time.perf_counter()
        for _ in range(num_runs):
            current = logits.clone()
            for proc in processors:
                for b in range(batch_size):
                    current[b] = proc(current[b], output_token_ids=prev_tokens)
        elapsed = time.perf_counter() - start
        results[name] = elapsed / num_runs * 1000  # ms
    
    return results


print(f"{'Config':<25} {'Latency (ms)':>15}")
print("=" * 42)
results = benchmark_sampling(vocab_size=32000, batch_size=8, num_runs=200)
for name, latency in results.items():
    print(f"{name:<25} {latency:>12.4f} ms")

print("\nNote: Sampling is typically <1% of total inference time.")
print("The model forward pass dominates; sampling overhead is negligible.")

## Exercises

### Assignment 1: Implement a Custom Logits Processor

Create a `TypicalSamplingProcessor` that implements *typical sampling* (Meister et al., 2023). Typical sampling keeps tokens whose information content (negative log-probability) is close to the expected information content (entropy).

In [ ]:
class TypicalSamplingProcessor(LogitsProcessor):
    """Typical sampling: keep tokens whose surprisal is close to the entropy.
    
    Steps:
    1. Compute probabilities and entropy of the distribution
    2. Compute surprisal (negative log-prob) for each token
    3. Compute |surprisal - entropy| for each token
    4. Sort by this deviation
    5. Keep smallest set whose cumulative probability >= typical_p
    """
    def __init__(self, typical_p: float = 0.95):
        self.typical_p = typical_p
    
    def __call__(self, logits: torch.Tensor, **kwargs) -> torch.Tensor:
        # TODO: Implement typical sampling
        # Step 1: Compute probabilities
        # probs = ...
        
        # Step 2: Compute entropy H = -sum(p * log(p))
        # entropy = ...
        
        # Step 3: Compute surprisal for each token: -log(p)
        # surprisal = ...
        
        # Step 4: Compute deviation |surprisal - entropy|
        # deviation = ...
        
        # Step 5: Sort by deviation, keep cumulative prob >= typical_p
        # sorted_indices = ...
        # Filter logits, set rejected tokens to -inf
        
        raise NotImplementedError("Implement typical sampling")


# Test your implementation
def test_typical_sampling():
    logits = torch.tensor([5.0, 3.0, 2.0, 1.5, 1.0, 0.5, 0.1, -1.0, -2.0, -3.0])
    processor = TypicalSamplingProcessor(typical_p=0.9)
    result = processor(logits)
    probs = F.softmax(result, dim=-1)
    active = (probs > 1e-8).sum().item()
    
    print(f"Typical sampling (p=0.9): {active} tokens kept")
    print(f"Probabilities: {probs.numpy().round(4)}")
    assert active < len(logits), "Should filter some tokens"
    assert probs.sum().item() > 0.99, "Should sum to ~1"
    print("PASSED!")

# Uncomment when ready:
# test_typical_sampling()

### Assignment 2: Implement Mirostat Sampling

Implement Mirostat v2 sampling, which dynamically adjusts the sampling to maintain a target perplexity (surprise level). Instead of a fixed temperature or top-p, Mirostat adapts on the fly.

In [ ]:
class MirostatV2Sampler:
    """Mirostat v2 sampling: adaptive perplexity control.
    
    Maintains a target surprise level (tau) by dynamically adjusting
    the number of tokens to consider.
    
    Parameters:
    - tau: target surprise value (higher = more random)
    - eta: learning rate for adapting mu
    - mu: current threshold (initialized to 2 * tau)
    """
    def __init__(self, tau: float = 5.0, eta: float = 0.1):
        self.tau = tau
        self.eta = eta
        self.mu = 2 * tau  # Initial threshold
    
    def sample(self, logits: torch.Tensor) -> int:
        """Sample a token using Mirostat v2.
        
        Steps:
        1. Compute surprisal (-log2(p)) for each token
        2. Keep only tokens with surprisal <= mu
        3. Sample from the remaining tokens
        4. Update mu based on the surprise of the selected token
           mu = mu - eta * (surprise - tau)
        """
        # TODO: Implement Mirostat v2 sampling
        # Step 1: Compute probabilities and surprisal
        # probs = F.softmax(logits, dim=-1)
        # surprisal = -torch.log2(probs)
        
        # Step 2: Filter tokens by mu threshold
        # mask = surprisal <= self.mu
        # filtered_logits = ...
        
        # Step 3: Sample from filtered distribution
        # token_id = ...
        
        # Step 4: Update mu
        # observed_surprise = surprisal[token_id]
        # self.mu = self.mu - self.eta * (observed_surprise - self.tau)
        
        raise NotImplementedError("Implement Mirostat v2")


def test_mirostat():
    logits = torch.randn(100)
    logits[0] = 5.0
    logits[1] = 3.0
    
    sampler = MirostatV2Sampler(tau=5.0, eta=0.1)
    tokens = []
    for _ in range(20):
        step_logits = logits + torch.randn(100) * 0.3
        token = sampler.sample(step_logits)
        tokens.append(token)
    
    print(f"Generated tokens: {tokens}")
    print(f"Unique tokens: {len(set(tokens))}")
    print(f"Final mu: {sampler.mu:.4f} (target tau: {sampler.tau})")
    print("PASSED!")

# Uncomment when ready:
# test_mirostat()

### Assignment 3: Build a Complete Sampling Pipeline

Build an end-to-end sampling pipeline that supports all the features we covered. Test it with autoregressive generation using a simple bigram model.

In [ ]:
class CompleteSamplingPipeline:
    """A complete sampling pipeline combining all processors.
    
    Must support:
    1. Configurable LogitsProcessor chain
    2. Custom processors (user-provided)
    3. Greedy and random sampling
    4. Seeded reproducibility
    5. Logprobs computation
    6. Best-of-n selection
    """
    def __init__(self, params: SamplingParams, 
                 custom_processors: List[LogitsProcessor] = None):
        self.params = params
        self.processors = build_logits_processors(params)
        if custom_processors:
            self.processors.extend(custom_processors)
        self.sampler = Sampler()
        self.generated_tokens: List[int] = []
    
    def step(self, logits: torch.Tensor) -> SamplerOutput:
        """Process one generation step."""
        # TODO: Apply all processors in sequence
        # current = logits.clone()
        # for proc in self.processors:
        #     current = proc(current, output_token_ids=self.generated_tokens)
        
        # TODO: Sample token
        # result = self.sampler.sample(current, self.params)
        
        # TODO: Track generated tokens
        # self.generated_tokens.extend(result.token_ids)
        
        raise NotImplementedError("Implement step()")
    
    def generate(self, get_logits_fn: Callable, num_steps: int) -> List[int]:
        """Autoregressive generation loop."""
        # TODO: Call step() repeatedly for num_steps
        # For each step, call get_logits_fn(self.generated_tokens)
        # to simulate getting logits from a model
        
        raise NotImplementedError("Implement generate()")
    
    def generate_best_of_n(self, get_logits_fn, num_steps, n, best_of):
        """Generate best_of sequences and return top n."""
        # TODO: Run generate() best_of times with different seeds
        # Score each by cumulative log-probability
        # Return top n
        
        raise NotImplementedError("Implement best_of_n")


def test_pipeline():
    vocab_size = 20
    
    # Simple bigram model: logits depend on last token
    def bigram_logits(generated_tokens):
        logits = torch.randn(vocab_size) * 0.5
        if generated_tokens:
            # Boost the next token in sequence
            last = generated_tokens[-1]
            logits[(last + 1) % vocab_size] += 3.0
            logits[(last + 2) % vocab_size] += 2.0
        else:
            logits[0] += 3.0
        return logits
    
    params = SamplingParams(
        temperature=0.7, top_p=0.9, 
        repetition_penalty=1.2, seed=42
    )
    pipeline = CompleteSamplingPipeline(params)
    tokens = pipeline.generate(bigram_logits, num_steps=15)
    
    print(f"Generated: {tokens}")
    print(f"Unique: {len(set(tokens))}/{len(tokens)}")
    assert len(tokens) == 15
    print("PASSED!")

# Uncomment when ready:
# test_pipeline()

## Summary

Key takeaways from this chapter:

1. **Logits processing is a chain**: Penalties -> Temperature -> Top-k -> Top-p/Min-p -> Custom processors
2. **Temperature** controls randomness: 0 = greedy, <1 = focused, >1 = diverse
3. **Top-k** provides a hard cutoff; **Top-p** adapts to model confidence
4. **Min-p** is a simpler alternative to top-p: keeps tokens above min_p * max_prob
5. **Repetition penalties** come in three flavors: multiplicative (repetition), additive-count (frequency), additive-binary (presence)
6. **The Sampler class** handles both greedy and random sampling, with seed-based reproducibility
7. **Best-of-n** trades compute for quality by generating multiple candidates
8. **Structured output** hooks (JSON mode, grammar) integrate as custom logits processors
9. **Batched sampling** in vLLM processes multiple sequences with different params efficiently

**Next**: Chapter 3.10 covers the vLLM API Server and OpenAI-compatible endpoints.